# 03B — Microsoft Optical real-data realism test

**Question:** does the calibration-frozen, telemetry-only design behave sensibly on real optical-network telemetry?

This notebook is intentionally **not an accuracy evaluation**. The public release contains no incident labels and Microsoft removed outage days. It therefore reports data availability, score transfer and unreviewed operational workload—not recall, precision, false-positive rate or localisation accuracy.

One policy is frozen before looking at development or holdout: per-channel robust references, segment-relative evidence, the 99th percentile of calibration daily maxima, two consecutive observations, two recovery observations and a one-hour case gap.

## 1. Setup and truth guard

In [ ]:
import gc
import os
import shutil
import sys
import tempfile
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import chi2

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

default_data_root = (Path("/content/drive/MyDrive/anomaly_detection")
                     if IN_COLAB else Path.home() / "anomaly_detection_data")
DATA_ROOT = Path(os.getenv("ANOMALY_DATA_ROOT")
                 or os.getenv("ANOMALY_DRIVE_ROOT")
                 or default_data_root).expanduser()
default_code_root = (DATA_ROOT / "research" / "milestone1" if IN_COLAB
                     else Path.cwd() if (Path.cwd() / "milestone1_core.py").is_file()
                     else Path.cwd() / "notebooks" / "drive_research")
NOTEBOOK_HOME = Path(os.getenv("ANOMALY_NOTEBOOK_HOME", default_code_root)).expanduser()
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

from milestone1_core import CORE_VERSION, new_output_directory, read_json, write_json

SECTOR = "microsoft_optical"
CANONICAL_RUN_ID = os.getenv("CANONICAL_RUN_ID", "microsoft_optical_core_v0_11_run1")
RUN_ROOT = DATA_ROOT / "outputs" / "canonical" / f"v{CORE_VERSION}" / SECTOR / CANONICAL_RUN_ID
CORE_ROOT = RUN_ROOT / "SPEC-CORE"
SPLIT_ROOT = RUN_ROOT / "SPLITS"
REALISM_VERSION = "1.0.0"
REALISM_RUN_ID = os.getenv("REALISM_RUN_ID", "microsoft_optical_realism_v1_run1")
OUTPUT_ROOT = DATA_ROOT / "outputs" / "realism" / f"v{REALISM_VERSION}" / SECTOR / REALISM_RUN_ID

if not CORE_ROOT.is_dir():
    raise FileNotFoundError(f"Run 01B for microsoft_optical first: {CORE_ROOT}")
if (RUN_ROOT / "SPEC-EVAL").exists():
    raise ValueError("Microsoft realism must not use SPEC-EVAL")

POLICY = {
    "threshold_quantile": 0.99,
    "minimum_consecutive": 2,
    "recovery_consecutive": 2,
    "case_gap_seconds": 3600,
    "minimum_segment_channels": 8,
    "reference_scale_floor_fraction": 0.10,
}
manifest = read_json(CORE_ROOT / "manifest.json")
catalogue = pd.read_parquet(CORE_ROOT / "metric_catalogue.parquet")
topology = pd.read_parquet(CORE_ROOT / "topology_memberships.parquet")
partitions = pd.read_parquet(SPLIT_ROOT / "time_partitions.parquet")
METRICS = catalogue["metric_id"].astype(str).tolist()
CADENCE_SECONDS = int(catalogue["expected_cadence_seconds"].dropna().unique()[0])
if catalogue["expected_cadence_seconds"].dropna().nunique() != 1:
    raise ValueError("This test expects one common optical cadence")
segments = topology.loc[topology["group_type"].eq("optical_segment"), ["entity_id", "group_id"]]
segments = segments.rename(columns={"group_id": "segment_id"}).drop_duplicates()
segment_sizes = segments.groupby("segment_id")["entity_id"].nunique()
if segment_sizes.min() < POLICY["minimum_segment_channels"]:
    raise ValueError("The selected fixture has too few peers in at least one segment")

display(pd.Series({
    "canonical_root": str(RUN_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "entities": segments["entity_id"].nunique(),
    "segments": segments["segment_id"].nunique(),
    "metrics": len(METRICS),
    "cadence_seconds": CADENCE_SECONDS,
    "evaluation_truth": "absent by design",
}, name="value").to_frame())
display(pd.Series(POLICY, name="frozen_value").to_frame())

## 2. Load one chronological partition at a time

Only `measured` values enter the reference or score. Missing and invalid observations remain missing; this notebook does not interpolate them. The long canonical table is pivoted only for the selected partition, keeping memory bounded.

In [ ]:
telemetry_glob = str(CORE_ROOT / "telemetry" / "part-*.parquet").replace("'", "''")
if any(not name.replace("_", "").isalnum() for name in METRICS):
    raise ValueError("Metric identifiers are not safe SQL names")
metric_columns = ",\n".join(
    f"max(CASE WHEN metric_id = '{name}' AND quality_code = 'measured' THEN value END) AS \"{name}\""
    for name in METRICS
)

def partition_bounds(name):
    row = partitions.loc[partitions["partition"].eq(name)]
    if len(row) != 1:
        raise ValueError(f"Expected one {name} time partition")
    return pd.to_datetime(row.iloc[0]["start_ts"], utc=True), pd.to_datetime(row.iloc[0]["end_ts"], utc=True)

def load_partition(name):
    start, end = partition_bounds(name)
    query = f"""
        SELECT event_ts, entity_id, episode_id, {metric_columns}
        FROM read_parquet('{telemetry_glob}')
        WHERE event_ts >= ? AND event_ts < ?
        GROUP BY event_ts, entity_id, episode_id
        ORDER BY entity_id, event_ts
    """
    with duckdb.connect() as connection:
        frame = connection.execute(query, [start, end]).df()
    frame["event_ts"] = pd.to_datetime(frame["event_ts"], utc=True)
    return frame.merge(segments, on="entity_id", how="left", validate="many_to_one")

calibration = load_partition("calibration")
if calibration.empty:
    raise ValueError("Calibration partition is empty")
print(f"Calibration rows: {len(calibration):,}")
display(calibration.head())

## 3. Fit the calibration-only reference

Each channel and metric receives a calibration median and robust scale (IQR / 1.349). A small pooled scale floor prevents a nearly constant calibration signal from producing unbounded scores. The reference is never updated from development or holdout.

In [ ]:
def robust_reference(frame):
    values = frame.set_index("entity_id")[METRICS]
    centre = values.groupby(level=0).median()
    q25 = values.groupby(level=0).quantile(0.25)
    q75 = values.groupby(level=0).quantile(0.75)
    scale = (q75 - q25) / 1.349
    pooled_scale = (frame[METRICS].quantile(0.75) - frame[METRICS].quantile(0.25)) / 1.349
    floor = (pooled_scale * POLICY["reference_scale_floor_fraction"]).clip(lower=1e-6)
    scale = scale.clip(lower=floor, axis=1).fillna(floor)
    return centre, scale

reference_centre, reference_scale = robust_reference(calibration)
reference_table = (reference_centre.stack().rename("centre").to_frame()
                   .join(reference_scale.stack().rename("scale")).reset_index()
                   .rename(columns={"level_1": "metric_id"}))
display(reference_table.head(12))

## 4. Three interpretable evidence channels

- `entity_change`: largest absolute departure from that channel's frozen reference.
- `peer_deviation`: largest departure from the simultaneous segment median.
- `segment_common_mode`: strongest shared segment movement, scaled by available channel count.

These are evidence channels, not fault classifiers. The segment median includes the target channel; with at least eight channels it remains robust, but this approximation is recorded and should be replaced by leave-one-out peers in a production scorer.

In [ ]:
ENTITY_CHANNELS = ["entity_change", "peer_deviation"]
GROUP_CHANNEL = "segment_common_mode"
ALL_CHANNELS = [*ENTITY_CHANNELS, GROUP_CHANNEL]

def score_partition(frame):
    entities = frame["entity_id"].astype(str)
    centre = reference_centre.reindex(entities).set_axis(frame.index)
    scale = reference_scale.reindex(entities).set_axis(frame.index)
    residuals = (frame[METRICS] - centre) / scale
    keys = [frame["event_ts"], frame["segment_id"]]
    segment_median = residuals.groupby(keys).transform("median")
    segment_count = residuals.groupby(keys).transform("count")

    entity_scores = frame[["event_ts", "entity_id", "episode_id", "segment_id"]].copy()
    entity_scores["entity_change"] = residuals.abs().max(axis=1, skipna=True)
    peer_evidence = (residuals - segment_median).abs().where(
        segment_count.ge(POLICY["minimum_segment_channels"])
    )
    entity_scores["peer_deviation"] = peer_evidence.max(axis=1, skipna=True)

    common_evidence = segment_median.abs().mul(np.sqrt(segment_count)).where(
        segment_count.ge(POLICY["minimum_segment_channels"])
    )
    group_scores = frame[["event_ts", "segment_id"]].copy()
    group_scores[GROUP_CHANNEL] = common_evidence.max(axis=1, skipna=True)
    group_scores = group_scores.groupby(["event_ts", "segment_id"], as_index=False)[GROUP_CHANNEL].max()
    return entity_scores, group_scores

calibration_entity_scores, calibration_group_scores = score_partition(calibration)

def daily_maximum_threshold(frame, channel, identity):
    blocks = frame[["event_ts", identity, channel]].dropna().copy()
    blocks["date"] = blocks["event_ts"].dt.floor("D")
    maxima = blocks.groupby([identity, "date"])[channel].max()
    if maxima.empty:
        raise ValueError(f"No calibration blocks for {channel}")
    return float(maxima.quantile(POLICY["threshold_quantile"])), len(maxima)

threshold_rows = []
for channel in ENTITY_CHANNELS:
    threshold, blocks = daily_maximum_threshold(calibration_entity_scores, channel, "entity_id")
    threshold_rows.append((channel, threshold, blocks, "entity_day_maximum"))
threshold, blocks = daily_maximum_threshold(calibration_group_scores, GROUP_CHANNEL, "segment_id")
threshold_rows.append((GROUP_CHANNEL, threshold, blocks, "segment_day_maximum"))
thresholds = pd.DataFrame(threshold_rows, columns=["channel", "threshold", "calibration_blocks", "threshold_block"])
thresholds["quantile"] = POLICY["threshold_quantile"]
display(thresholds)

## 5. Convert evidence into alerts and cases

Two consecutive 15-minute exceedances open an alert. Two consecutive recoveries close it. Alerts from different channels on the same scope are merged when separated by at most one hour. These are **unreviewed cases**; without incident truth they must not be called false positives.

In [ ]:
def alert_intervals(frame, channel, identity, scope_type, partition):
    threshold = thresholds.set_index("channel").loc[channel, "threshold"]
    alerts = []
    for scope_id, group in frame[["event_ts", identity, channel]].dropna().groupby(identity, sort=True):
        group = group.sort_values("event_ts")
        active = False
        above_run = below_run = 0
        start = last_above = peak_ts = candidate_start = None
        peak_score = candidate_peak = -np.inf
        candidate_peak_ts = None
        previous_ts = None
        for row in group.itertuples(index=False):
            timestamp, score = row.event_ts, float(getattr(row, channel))
            if previous_ts is not None and timestamp - previous_ts > pd.Timedelta(seconds=CADENCE_SECONDS * 1.5):
                if active:
                    alerts.append((partition, channel, scope_type, str(scope_id), start, last_above, peak_ts, peak_score))
                active = False
                above_run = below_run = 0
                peak_score = candidate_peak = -np.inf
            if not active and score >= threshold:
                if above_run == 0:
                    candidate_start, candidate_peak_ts, candidate_peak = timestamp, timestamp, score
                elif score > candidate_peak:
                    candidate_peak_ts, candidate_peak = timestamp, score
                above_run += 1
                if above_run >= POLICY["minimum_consecutive"]:
                    active, start, last_above = True, candidate_start, timestamp
                    peak_ts, peak_score = candidate_peak_ts, candidate_peak
            elif active and score >= threshold:
                below_run, last_above = 0, timestamp
                if score > peak_score:
                    peak_ts, peak_score = timestamp, score
            elif active:
                below_run += 1
                if below_run >= POLICY["recovery_consecutive"]:
                    alerts.append((partition, channel, scope_type, str(scope_id), start, last_above, peak_ts, peak_score))
                    active = False
                    above_run = below_run = 0
                    peak_score = candidate_peak = -np.inf
            else:
                above_run = 0
                candidate_peak = -np.inf
            previous_ts = timestamp
        if active:
            alerts.append((partition, channel, scope_type, str(scope_id), start, last_above, peak_ts, peak_score))
    return alerts

ALERT_COLUMNS = ["partition", "channel", "scope_type", "scope_id", "alert_start", "alert_end", "peak_ts", "peak_score"]

def consolidate_cases(alerts):
    rows = []
    maximum_gap = pd.Timedelta(seconds=POLICY["case_gap_seconds"])
    for keys, group in alerts.groupby(["partition", "scope_type", "scope_id"], sort=True):
        current = None
        for alert in group.sort_values("alert_start").itertuples(index=False):
            if current is None or alert.alert_start > current["case_end"] + maximum_gap:
                if current is not None:
                    rows.append(current)
                current = {"partition": keys[0], "scope_type": keys[1], "scope_id": keys[2],
                           "case_start": alert.alert_start, "case_end": alert.alert_end,
                           "peak_score": alert.peak_score, "channels": {alert.channel}, "alert_count": 1}
            else:
                current["case_end"] = max(current["case_end"], alert.alert_end)
                current["peak_score"] = max(current["peak_score"], alert.peak_score)
                current["channels"].add(alert.channel)
                current["alert_count"] += 1
        if current is not None:
            rows.append(current)
    case_columns = ["case_id", "partition", "scope_type", "scope_id", "case_start", "case_end", "peak_score", "channels", "alert_count"]
    cases = pd.DataFrame(rows)
    if cases.empty:
        return pd.DataFrame(columns=case_columns)
    else:
        cases["channels"] = cases["channels"].map(lambda values: ",".join(sorted(values)))
        cases.insert(0, "case_id", [f"MS-CASE-{number:06d}" for number in range(1, len(cases) + 1)])
    return cases[case_columns]

def score_summary(entity_scores, group_scores, partition):
    rows = []
    for channel in ENTITY_CHANNELS:
        values = entity_scores[channel].dropna()
        rows.append((partition, channel, len(values), values.median(), values.quantile(0.95), values.quantile(0.99)))
    values = group_scores[GROUP_CHANNEL].dropna()
    rows.append((partition, GROUP_CHANNEL, len(values), values.median(), values.quantile(0.95), values.quantile(0.99)))
    return rows

all_alerts = []
distribution_rows = []
exposure_rows = []
q_factor_rows = []
representative_series = None
for partition in ["calibration", "development", "holdout"]:
    frame = calibration if partition == "calibration" else load_partition(partition)
    entity_scores, group_scores = (
        (calibration_entity_scores, calibration_group_scores)
        if partition == "calibration" else score_partition(frame)
    )
    rows = []
    for channel in ENTITY_CHANNELS:
        rows.extend(alert_intervals(entity_scores, channel, "entity_id", "optical_channel", partition))
    rows.extend(alert_intervals(group_scores, GROUP_CHANNEL, "segment_id", "optical_segment", partition))
    all_alerts.extend(rows)
    distribution_rows.extend(score_summary(entity_scores, group_scores, partition))
    observed_days = frame.assign(date=frame["event_ts"].dt.floor("D")).drop_duplicates(["entity_id", "date"]).shape[0]
    exposure_rows.append((partition, observed_days, len(frame)))
    q_values = frame["q_factor"].dropna()
    q_factor_rows.append((partition, len(q_values), int(q_values.lt(6.5).sum()), q_values.min(), q_values.median()))
    if partition == "development":
        first_segment = sorted(frame["segment_id"].dropna().unique())[0]
        representative_series = frame.loc[frame["segment_id"].eq(first_segment), ["event_ts", "entity_id", "q_factor"]].copy()
    if partition != "calibration":
        del frame, entity_scores, group_scores
        gc.collect()

alerts = pd.DataFrame(all_alerts, columns=ALERT_COLUMNS).sort_values(["partition", "alert_start"]).reset_index(drop=True)
alerts.insert(0, "alert_id", [f"MS-ALERT-{number:06d}" for number in range(1, len(alerts) + 1)])
cases = consolidate_cases(alerts)
score_distribution = pd.DataFrame(distribution_rows, columns=["partition", "channel", "score_rows", "p50", "p95", "p99"])
exposure = pd.DataFrame(exposure_rows, columns=["partition", "observed_entity_days", "wide_rows"])
q_factor_reference = pd.DataFrame(q_factor_rows, columns=["partition", "observations", "q_factor_below_6_5", "minimum", "median"])

def poisson_rate_interval(count, exposure_value):
    lower = 0.0 if count == 0 else 0.5 * chi2.ppf(0.025, 2 * count)
    upper = 0.5 * chi2.ppf(0.975, 2 * (count + 1))
    return 100 * lower / exposure_value, 100 * upper / exposure_value

workload = exposure.copy()
workload["alerts"] = workload["partition"].map(alerts.groupby("partition").size()).fillna(0).astype(int)
workload["cases"] = workload["partition"].map(cases.groupby("partition").size()).fillna(0).astype(int)
workload["cases_per_100_observed_entity_days"] = 100 * workload["cases"] / workload["observed_entity_days"]
intervals = [poisson_rate_interval(row.cases, row.observed_entity_days) for row in workload.itertuples()]
workload["rate_95ci_lower"] = [value[0] for value in intervals]
workload["rate_95ci_upper"] = [value[1] for value in intervals]

display(workload)
display(score_distribution)
display(q_factor_reference)

## 6. Inspect and save the evidence

The Q-factor value 6.5 is shown only as a published operational reference. A crossing is not treated as a labelled fault, and its absence cannot validate the detector because outage days were removed.

In [ ]:
figure_cache = Path(tempfile.mkdtemp(prefix="microsoft-optical-figures-"))

if representative_series is not None and not representative_series.empty:
    chosen = sorted(representative_series["entity_id"].unique())[:6]
    daily = (representative_series.loc[representative_series["entity_id"].isin(chosen)]
             .set_index("event_ts").groupby("entity_id")["q_factor"].resample("D").median()
             .rename("q_factor").reset_index())
    fig, ax = plt.subplots(figsize=(12, 5))
    for entity_id, group in daily.groupby("entity_id"):
        ax.plot(group["event_ts"], group["q_factor"], label=entity_id, linewidth=1)
    ax.axhline(6.5, color="firebrick", linestyle="--", label="published Q reference (not truth)")
    ax.set(title="Development Q-factor: six channels in one segment", ylabel="daily median Q-factor")
    ax.legend(ncol=2, fontsize=8)
    fig.tight_layout()
    fig.savefig(figure_cache / "q_factor_segment_series.png", dpi=140)
    plt.show()
    plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4))
positions = np.arange(len(workload))
rates = workload["cases_per_100_observed_entity_days"].to_numpy()
lower = rates - workload["rate_95ci_lower"].to_numpy()
upper = workload["rate_95ci_upper"].to_numpy() - rates
ax.bar(positions, rates, color=["#4C78A8", "#F58518", "#54A24B"])
ax.errorbar(positions, rates, yerr=[lower, upper], fmt="none", color="black", capsize=4)
ax.set_xticks(positions, workload["partition"])
ax.set(ylabel="cases per 100 observed entity-days", title="Unreviewed operational workload")
fig.tight_layout()
fig.savefig(figure_cache / "case_workload.png", dpi=140)
plt.show()
plt.close(fig)

realism_manifest = {
    "realism_version": REALISM_VERSION,
    "sector": SECTOR,
    "canonical_run": str(RUN_ROOT),
    "canonical_fingerprint": manifest["fingerprint"],
    "status": "descriptive_only_no_incident_truth",
    "policy": POLICY,
    "limitations": [
        "The public release contains no incident labels and excludes outage days.",
        "Cases are unreviewed workload, not false positives.",
        "Segment-median peer evidence includes the target channel in this lightweight test.",
        "No localisation accuracy can be estimated.",
    ],
    "output_rows": {
        "thresholds": len(thresholds), "workload": len(workload),
        "score_distribution": len(score_distribution),
        "alerts": len(alerts), "cases": len(cases),
    },
}

with new_output_directory(OUTPUT_ROOT) as output:
    thresholds.to_csv(output / "calibration_thresholds.csv", index=False)
    workload.to_csv(output / "workload_by_partition.csv", index=False)
    score_distribution.to_csv(output / "score_distribution.csv", index=False)
    q_factor_reference.to_csv(output / "q_factor_reference.csv", index=False)
    reference_table.to_parquet(output / "calibration_reference.parquet", index=False)
    alerts.to_parquet(output / "alerts.parquet", index=False)
    cases.to_parquet(output / "cases.parquet", index=False)
    shutil.copytree(figure_cache, output / "figures")
    write_json(output / "realism_manifest.json", realism_manifest)

shutil.rmtree(figure_cache)
print("Saved:", OUTPUT_ROOT)
print("Conclusion boundary: real-data behaviour assessed; detection accuracy remains unknown.")